# DS1 KPI payloads (DS3 supplement)

## Overview
DS3 nulls the T1, T2, fRB and fCX fits it rejects and keeps no errors before 2025-07, but the DS1 payloads behind those values still hold the whole fit ([Phase 01, B3](../../docs/phases/01_eda.md#b3-ds1-measurements)). So we read them back and add them to the KPI table as `kpi_supplemented`, which is what B4 profiles. The DS3 side is in [datasheet 3](../../docs/datasheets/03_benchmarking_files.md).

**Question.** Do the payload fits match DS3 where both exist, how much of the KPI table can they fill, and how does the scatter between RB seeds compare with the fit error?

**Method.** The work happens in `qpu_drift.derived.supplement` (CLI `supplement-kpi`) - this notebook just checks what it writes. We follow each `output_info` pointer through the guard to a local payload (section 1), check the payload values, errors and stored fits against DS3 (section 2), and measure how much of each metric and era the new columns fill (section 3). Then we compare the RB seed scatter with the fit error (section 4) and check the strata columns B4 splits by (section 5).


In [1]:
import numpy as np
import pandas as pd

from qpu_drift import periods
from qpu_drift.derived.supplement import (
    PAYLOAD_FITS,
    PAYLOAD_LINKS,
    RB_TYPES,
    SUPPLEMENT_COLS,
    SUPPLEMENT_METRICS,
)
from qpu_drift.sources.benchmarking.load import (
    DERIVED_DIR,
    KPI_SUPPLEMENTED_PATH,
    load_kpi_l2,
    load_runs_l2,
)
from qpu_drift.sources.benchmarking.recover import DEFAULT_SPINE

pd.set_option("display.max_rows", 300, "display.max_columns", 40, "display.width", 250, "display.max_colwidth", 80)
l2 = load_kpi_l2()
kpi = load_kpi_l2(KPI_SUPPLEMENTED_PATH)
links = pd.read_parquet(DERIVED_DIR / PAYLOAD_LINKS)
fits = pd.read_parquet(DERIVED_DIR / PAYLOAD_FITS)

keys = ["created_at", "element", "metric"]
a, b = l2.set_index(keys).sort_index(), kpi.set_index(keys).sort_index()
print(f"rows: kpi_enriched {len(a):,}, kpi_supplemented {len(b):,}, same keys {a.index.equals(b.index)}")
print("columns only in kpi_supplemented:", [c for c in b.columns if c not in a.columns])
print("kpi_enriched columns missing from kpi_supplemented:", [c for c in a.columns if c not in b.columns])
same = {c: bool(a[c].equals(b[c])) for c in a.columns if c != "result_id"}
print("DS3 columns identical:", same)
had = a["result_id"].notna()
print(f"result_id: {int(had.sum()):,} present in kpi_enriched, all unchanged {bool((a.loc[had, 'result_id'] == b.loc[had, 'result_id']).all())}; "
      f"{int((~had & b['result_id'].notna()).sum()):,} filled")

era = load_runs_l2().set_index("created_at")["gating_regime"]
kpi["era"] = kpi["created_at"].map(era)
sub = kpi[kpi["metric"].isin(SUPPLEMENT_METRICS)].copy()
print("supplement columns missing:", [c for c in SUPPLEMENT_COLS if c not in kpi.columns])
print(f"T1/T2/fRB/fCX rows {len(sub):,}, runs {kpi['created_at'].nunique():,}")
print(f"pointers {len(links):,}, payload fit rows {len(fits):,} from {fits['result_id'].nunique():,} payloads")
print("rows by era:", sub["era"].value_counts(dropna=False).to_dict())

rows: kpi_enriched 104,800, kpi_supplemented 104,800, same keys True
columns only in kpi_supplemented: ['payload_value', 'payload_error', 'payload_r2', 'seed_error', 'rb_lengths', 'error_source', 'in_incident', 'coupling_retired']
kpi_enriched columns missing from kpi_supplemented: []
DS3 columns identical: {'element_type': True, 'value': True, 'error': True, 'r2': True, 'value_status': True, 'value_source': True, 'measured_at': True}
result_id: 37,717 present in kpi_enriched, all unchanged True; 32,512 filled
supplement columns missing: []
T1/T2/fRB/fCX rows 83,840, runs 655
pointers 41,433, payload fit rows 60,576 from 7,428 payloads
rows by era: {'era3_postproc': 40576, 'era1_filter': 28544, 'era0_ungated': 9600, 'era2_ungated': 5120}


## 1. Pointer to payload

`supplement-kpi` reads `kpi_enriched` and never rewrites it, so the setup cell above first checks that `kpi_supplemented` is the same table plus the new columns. Now we follow each `output_info` result id for T1, T2, fRB and fCX to its payload. A pointer is *guarded* when its spine row has the metric's producer type and a `time_start` near the run's `measured_at` (`recover.resolve_and_guard`), and *local* when its file is in one of the two fetch roots. T1 and T2 pointers are per group, so one covers several qubits; fRB and fCX pointers are per element. RB2q is only local for the weekly 3a sample and the `decode-ds1` rows.


In [2]:
links["local"] = links["path"].notna()
links["era"] = links["measured_at"].map(sub.groupby("measured_at")["era"].first()).fillna("no DS3 run")
decoded = fits.groupby("metric")["result_id"].nunique()
chain = links.groupby("metric").agg(
    pointers=("result_id", "size"), guarded=("guarded", "sum"), local=("local", "sum"),
    local_payloads=("path", "nunique"))
chain["decoded_payloads"] = decoded
chain["targets"] = fits.groupby("metric").size()
chain["targets_with_value"] = fits.groupby("metric")["value"].count()
chain["targets_with_fit"] = fits.groupby("metric")["x0"].count()
display(chain)
print("local share of guarded pointers, by metric and era:")
display(links[links["guarded"]].pivot_table(index="metric", columns="era", values="local", aggfunc="mean", dropna=False).round(3))
nods3 = links.loc[links["era"] == "no DS3 run", "measured_at"].drop_duplicates()
print(f"output_info starts with no DS3 run: {len(nods3)}, {int((nods3 < kpi['created_at'].min()).sum())} before the first DS3 run")
print("unguarded pointers, by metric and first/last measured_at:")
display(links[~links["guarded"]].groupby("metric")["measured_at"].agg(["size", "min", "max"]))
print("measurement types in the fits:", fits["measurement_type"].value_counts().to_dict())
print("fit targets with no metric/element mapping:", int(fits["metric"].isna().sum() + fits["element"].isna().sum()))


,pointers,guarded,local,local_payloads,decoded_payloads,targets,targets_with_value,targets_with_fit
metric,,,,,,,,
T1,1317,1201,1201,1201,1201,19209,19209,19209
T2,1317,1201,1201,1201,1201,19209,19209,19209
fCX,19947,18423,3884,3884,3884,3884,3881,3881
fRB,18852,18244,18244,1142,1142,18274,18274,18274


local share of guarded pointers, by metric and era:


era,era1_filter,era2_ungated,era3_postproc,no DS3 run
metric,,,,
T1,1.000,1.00,1.000,1.000
T2,1.000,1.00,1.000,1.000
fCX,0.161,0.15,0.251,0.258
fRB,1.000,1.00,1.000,1.000


output_info starts with no DS3 run: 94, 53 before the first DS3 run
unguarded pointers, by metric and first/last measured_at:


,size,min,max
metric,,,
T1,116,2024-04-15 08:51:34,2024-05-18 18:10:05
T2,116,2024-04-15 08:51:34,2024-05-18 18:10:05
fCX,1524,2024-04-15 08:51:34,2025-08-22 08:26:52
fRB,608,2024-04-15 03:14:36,2024-05-18 18:10:05


measurement types in the fits: {'T1': 19209, 'T2echo': 19209, 'RandomizedBenchmarking1q': 18274, 'RandomizedBenchmarking2q': 3884}
fit targets with no metric/element mapping: 0


**Results.**

- `kpi_supplemented` has the same 104,800 rows as `kpi_enriched`, and every DS3 column is identical. The 37,717 `result_id` values already there are unchanged, and we fill 32,512 more.
- Every guarded T1, T2 and fRB pointer is local, but only 0.15 to 0.25 of the fCX ones per era - that's the weekly RB2q sample.
- Every local payload decodes, and every target maps to a metric and element. All have a value and a fit except 3 fCX targets.
- The unguarded pointers are all from 2024-04-15 to 2024-05-18, before DS3 starts, plus fCX up to 2025-08-22 08:26:52 - the run 3a found with its DS1 rows 735 min early.
- The "no DS3 run" column is the 94 `output_info` starts DS3 has no run for, 53 of them before DS3's first run.


## 2. Payload against DS3

Does the payload say what DS3 says? We compare `payload_value`, `payload_error` and `payload_r2` with DS3's `value`, `error` and `r2` where both exist, by `value_source`, using the relative difference $|a - b| / |b|$. `decode-ds1` filled the `ds1` rows from these same payloads, so only the `qcaas` and `output_info` rows are an independent check.

We also check we read the stored RB fits right. The fit is

$$y = A\,e^{-x/x_0} + B$$

and the producer turns the decay constant into a fidelity,

$$F_\text{1q} = \tfrac12 + \tfrac12\,e^{-1/x_0}, \qquad F_\text{CX} = \tfrac14 + \tfrac34\,e^{-1/(1.5\,x_0)}$$

so we compare that (`x0_value`) with the payload's own fidelity, recompute r2 from the stored fit (`calc_r2`), and check the seed mean of `raw_output` is the stored curve (`curve_match`). Last, we refit the stored curve ourselves (`refit_x0`): a grid search over $x_0$ within 100× of the stored one, with $A$ and $B$ solved exactly at each $x_0$. It's NaN when the stored $x_0$ isn't positive or the best $x_0$ lands on the edge of the grid.


In [3]:
def rel_diff(a: pd.Series, b: pd.Series) -> pd.Series:
    """|a - b| / |b| on the rows where both are present."""
    both = a.notna() & b.notna()
    return ((a[both] - b[both]).abs() / b[both].abs()).replace(np.inf, np.nan)


rows = []
for (metric, source), g in sub.groupby(["metric", "value_source"]):
    for ds3, pay in (("value", "payload_value"), ("error", "payload_error"), ("r2", "payload_r2")):
        d = rel_diff(g[pay], g[ds3])
        rows.append((metric, source, ds3, len(d), int((d == 0).sum()), int((d > 1e-9).sum()),
                     d.median() if len(d) else np.nan, d.max() if len(d) else np.nan))
cmp = pd.DataFrame(rows, columns=["metric", "value_source", "column", "n_both", "n_equal", "n_rel_gt_1e-9", "median_rel", "max_rel"])
display(cmp[cmp["n_both"] > 0])

bad = sub[(rel_diff(sub["payload_value"], sub["value"]) > 1e-9).reindex(sub.index, fill_value=False)]
print(f"rows with payload_value != value (rel > 1e-9): {len(bad)}")
display(bad[["created_at", "metric", "element", "value_source", "value_status", "value", "payload_value", "result_id"]].head(20))

rb = fits[fits["measurement_type"].isin(RB_TYPES)]
print("x0_value vs payload fidelity, max rel diff by metric:", rel_diff(rb["x0_value"], rb["value"]).groupby(rb["metric"]).max().to_dict())
print("payload_r2 - calc_r2 (0 by construction where no r2 is stored), max abs by metric:",
      (fits["payload_r2"] - fits["calc_r2"]).abs().groupby(fits["metric"]).max().to_dict())
print("curve_match by metric:", rb.groupby("metric")["curve_match"].value_counts(dropna=False).to_dict())
refit = rel_diff(rb["refit_x0"], rb["x0"])
print("refit_x0 vs stored x0, rel diff quantiles by metric:")
display(refit.groupby(rb["metric"]).quantile([0.5, 0.99, 1]).unstack())
print("refit NaN (no interior minimum) by metric:", rb["refit_x0"].isna().groupby(rb["metric"]).sum().to_dict(),
      "| of those, stored x0 < 0:", (rb["refit_x0"].isna() & (rb["x0"] < 0)).groupby(rb["metric"]).sum().to_dict())
off = rb.loc[refit[refit > 1e-3].index]
print(f"refits more than 0.1% from the stored x0: {len(off)}; refit_r2 > calc_r2 in {int((off['refit_r2'] > off['calc_r2']).sum())}")
display(off[["metric", "target", "result_id", "x0", "refit_x0", "calc_r2", "refit_r2", "value"]].head(20))

,metric,value_source,column,n_both,n_equal,n_rel_gt_1e-9,median_rel,max_rel
0,T1,ds1,value,855,855,0,0.0,0.000000e+00
1,T1,ds1,error,855,855,0,0.0,0.000000e+00
2,T1,ds1,r2,855,855,0,0.0,0.000000e+00
6,T1,output_info,value,256,256,0,0.0,0.000000e+00
8,T1,output_info,r2,256,192,1,0.0,4.144754e-08
9,T1,qcaas,value,14366,14366,0,0.0,0.000000e+00
10,T1,qcaas,error,9976,9976,0,0.0,0.000000e+00
11,T1,qcaas,r2,14200,13947,0,0.0,1.581515e-14
12,T2,ds1,value,2563,2563,0,0.0,0.000000e+00
13,T2,ds1,error,2563,2560,0,0.0,0.000000e+00


rows with payload_value != value (rel > 1e-9): 0


,created_at,metric,element,value_source,value_status,value,payload_value,result_id


x0_value vs payload fidelity, max rel diff by metric: {'fCX': 1.9151199781502121e-16, 'fRB': 1.6115509087657384e-16}
payload_r2 - calc_r2 (0 by construction where no r2 is stored), max abs by metric: {'T1': 5.188072194073357e-13, 'T2': 8.344436253082677e-13, 'fCX': 6.292744103575387e-13, 'fRB': 9.845457782375888e-13}
curve_match by metric: {('fCX', True): 3881, ('fCX', nan): 3, ('fRB', True): 18274}
refit_x0 vs stored x0, rel diff quantiles by metric:


,0.50,0.99,1.00
metric,,,
fCX,4.909802e-07,85.087112,98.701272
fRB,1.283204e-06,0.989659,97.160092


refit NaN (no interior minimum) by metric: {'fCX': 446, 'fRB': 224} | of those, stored x0 < 0: {'fCX': 361, 'fRB': 21}
refits more than 0.1% from the stored x0: 501; refit_r2 > calc_r2 in 494


,metric,target,result_id,x0,refit_x0,calc_r2,refit_r2,value
75,fRB,Q23,218827,2.742710e+04,2.748605e+04,0.982519,0.982519,0.999982
79,fRB,Q33,218827,4.508306e+05,4.352416e+07,0.604723,0.604730,0.999999
178,fRB,Q33,219490,1.540066e+06,7.457370e+07,0.505215,0.505227,1.000000
190,fRB,Q26,219491,1.536231e+00,2.023389e-02,0.613983,0.613983,0.760777
537,fRB,Q25,222677,1.930749e+06,1.225149e+08,0.865375,0.865388,1.000000
550,fRB,Q22,222678,1.561534e+00,1.130702e-01,0.732483,0.732483,0.763542
557,fCX,C14-T15,222701,1.377439e+02,1.374948e+02,0.048103,0.048103,0.996379
638,fRB,Q33,223346,1.113085e+07,2.019272e+08,0.953858,0.953859,1.000000
799,fRB,Q26,224684,1.465382e+00,1.571432e-02,0.526374,0.526374,0.752697
903,fCX,C14-T15,225376,7.851780e+04,4.016257e+06,0.002372,0.002372,0.999994


**Results.**

- On the independent rows `payload_value` equals `value` everywhere - no row of any source differs by more than $10^{-9}$. `payload_error` equals `error` on every `qcaas` row (`output_info` rows have no error), and `payload_r2` is within 4.1e-08 of `r2`.
- We read the fits right: `x0_value` matches the payload fidelity to about 1e-16, `calc_r2` matches the stored r2 to 1e-12, and the seed mean is the stored curve for every RB target with seeds (3 fCX have none).
- Our refit lands on the stored $x_0$ to a median of about 1e-06. It's NaN for 446 fCX targets, mostly where the stored $x_0$ is negative (361), and for 224 fRB (21 negative).
- 501 refits sit more than 0.1% off, but there $x_0$ just isn't pinned down - in the 20 shown the r2 agrees to 1.3e-05. They're near-flat decays with fidelity ≈ 1, or curves that are flat from the first length (fRB on Q22 and Q26, fidelity 0.75 to 0.88).


## 3. Coverage

Next, how much each new column fills, by metric and gating era. Shares are over every row of the metric, nulls included. A row gets payload columns when its run's `output_info` start, metric and element match a guarded local pointer (T1 and T2 take the element from the payload's targets). The `output_info` cache ends at 2026-08-17 20:06:09, so later runs can't match - we show era 3 up to then as well. Then come `error_source`, fCX coverage per coupling (B4 uses it to decide on fetching the rest of RB2q), and why each row without a payload has none.


In [4]:
cols = ["result_id", "error", "payload_value", "payload_error", "payload_r2", "seed_error"]
cov = sub.groupby(["metric", "era"], dropna=False)[cols].agg(lambda s: s.notna().mean()).round(3)
cov.insert(0, "rows", sub.groupby(["metric", "era"], dropna=False).size())
display(cov)
print("error_source by metric and era:")
display(pd.crosstab([sub["metric"], sub["era"]], sub["error_source"], dropna=False))

oi_end = links["measured_at"].max()
in_cache = sub[sub["created_at"] <= oi_end]
print(f"payload_value share by metric and era, runs up to the output_info cache end ({oi_end}):")
display(in_cache.groupby(["metric", "era"])["payload_value"].agg(lambda s: s.notna().mean()).unstack().round(3))

fcx = sub[sub["metric"] == "fCX"]
per = fcx.groupby("element").agg(rows=("value", "size"), payload_share=("payload_value", lambda s: s.notna().mean()))
print("fCX payload_value share per coupling:", per["payload_share"].describe().round(3).to_dict())
display(per.sort_values("payload_share").head(10))

sub["run_at"] = sub["created_at"].map(sub.groupby("created_at")["measured_at"].max())
miss = sub[sub["payload_value"].isna()].reset_index()
cand = pd.concat([
    miss.merge(links[links["element"].notna()], left_on=["run_at", "metric", "element"],
               right_on=["measured_at", "metric", "element"]),
    miss.merge(links[links["element"].isna()].drop(columns="element"), left_on=["run_at", "metric"],
               right_on=["measured_at", "metric"]),
]).groupby("index").agg(guarded=("guarded", "any"), local=("local", "any"))
m = miss.join(cand, on="index")
m["reason"] = np.select(
    [m["run_at"].isna() & (m["created_at"] > oi_end), m["run_at"].isna(), m["guarded"].isna(),
     ~m["guarded"].astype(bool), ~m["local"].astype(bool)],
    ["after output_info cache", "no output_info start", "no pointer", "unguarded", "not local"],
    "local payload, no value for target")
print(f"T1/T2/fRB/fCX rows with no payload_value: {len(m):,}")
display(pd.crosstab([m["metric"], m["era"]], m["reason"], dropna=False))
nop = m[m["reason"] == "no pointer"]
print("DS3 status of the 'no pointer' rows:", nop.groupby("metric")["value_status"].value_counts().to_dict(),
      "| with a DS3 value:", int(nop["value"].notna().sum()))


rows  result_id  error  payload_value  payload_error  payload_r2  seed_error
metric era                                                                                         
T1     era0_ungated    2400        0.0  0.000          0.000          0.000       0.000       0.000
       era1_filter     7136      0.995  0.000          0.995          0.995       0.995       0.000
       era2_ungated    1280        1.0  1.000          1.000          1.000       1.000       0.000
       era3_postproc  10144      0.945  0.989          0.945          0.945       0.945       0.000
T2     era0_ungated    2400        0.0  0.000          0.000          0.000       0.000       0.000
       era1_filter     7136      0.995  0.000          0.995          0.995       0.995       0.000
       era2_ungated    1280        1.0  1.000          1.000          1.000       1.000       0.000
       era3_postproc  10144      0.945  0.989          0.945          0.945       0.945       0.000
fCX    era0_ungated    2400        0.0  0.000          0.000          0.000       0.000       0.000
       era1_filter     7136      0.972  0.000          0.157          0.157       0.157       0.152
       era2_ungated    1280      0.972  0.972          0.145          0.145       0.145       0.141
       era3_postproc  10144      0.891  0.928          0.224          0.224       0.224       0.169
fRB    era0_ungated    2400        0.0  0.000          0.000          0.000       0.000       0.000
       era1_filter     7136      0.966  0.000          0.966          0.966       0.966       0.937
       era2_ungated    1280      0.897  0.897          0.897          0.897       0.897       0.855
       era3_postproc  10144      0.891  0.933          0.891          0.891       0.891       0.861

error_source by metric and era:


error_source            ds3  none  payload
metric era                                
T1     era0_ungated       0  2400        0
       era1_filter        0    34     7102
       era2_ungated    1280     0        0
       era3_postproc  10033    79       32
T2     era0_ungated       0  2400        0
       era1_filter        0    34     7102
       era2_ungated    1280     0        0
       era3_postproc  10033    79       32
fCX    era0_ungated       0  2400        0
       era1_filter        0  6019     1117
       era2_ungated    1244    36        0
       era3_postproc   9418   725        1
fRB    era0_ungated       0  2400        0
       era1_filter        0   241     6895
       era2_ungated    1148   132        0
       era3_postproc   9469   673        2

payload_value share by metric and era, runs up to the output_info cache end (2026-08-17 20:06:09):


era,era0_ungated,era1_filter,era2_ungated,era3_postproc
metric,,,,
T1,0.0,0.995,1.000,0.995
T2,0.0,0.995,1.000,0.995
fCX,0.0,0.157,0.145,0.235
fRB,0.0,0.966,0.897,0.938


fCX payload_value share per coupling: {'count': 35.0, 'mean': 0.164, 'std': 0.078, 'min': 0.0, '25%': 0.133, '50%': 0.134, '75%': 0.16, 'max': 0.421}


,rows,payload_share
element,,
25-17,4,0.000000
2-9,655,0.108397
19-11,655,0.109924
23-22,655,0.123664
13-14,655,0.128244
32-33,655,0.129771
32-31,655,0.131298
17-25,651,0.132104
4-5,655,0.132824


T1/T2/fRB/fCX rows with no payload_value: 27,260


reason                after output_info cache  local payload, no value for target  no output_info start  no pointer  not local
metric era                                                                                                                    
T1     era0_ungated                         0                                   0                  2400           0          0
       era1_filter                          0                                  34                     0           0          0
       era2_ungated                         0                                   0                     0           0          0
       era3_postproc                      512                                  17                    32           0          0
T2     era0_ungated                         0                                   0                  2400           0          0
       era1_filter                          0                                  34                     0           0          0
       era2_ungated                         0                                   0                     0           0          0
       era3_postproc                      512                                  17                    32           0          0
fCX    era0_ungated                         0                                   0                  2400           0          0
       era1_filter                          0                                   0                     0         198       5821
       era2_ungated                         0                                   0                     0          36       1058
       era3_postproc                      512                                   3                    32         562       6767
fRB    era0_ungated                         0                                   0                  2400           0          0
       era1_filter                          0                                   0                     0         241          0
       era2_ungated                         0                                   0                     0         132          0
       era3_postproc                      512                                   0                    32         564          0

DS3 status of the 'no pointer' rows: {('fCX', 'no_result'): 796, ('fRB', 'no_result'): 937} | with a DS3 value: 0


**Results.**

- From era 1 on, T1 and T2 are almost fully covered (0.995 to 1.000), fRB mostly (0.89 to 0.97) and fCX only 0.15 to 0.22 - the weekly RB2q sample again. Era 0 has no `output_info`, so nothing. Stopping era 3 at the cache end lifts it to 0.995 (T1, T2), 0.938 (fRB) and 0.235 (fCX).
- DS3 has no error in eras 0 and 1, so in era 1 the payload error is the only one: it fills 7,102 T1, 7,102 T2, 6,895 fRB and 1,117 fCX rows. That leaves 6,019 era-1 fCX rows, and all of era 0, with no error at all.
- Per coupling, fCX coverage sits tight around a median 0.134 (quartiles 0.133 and 0.160). Only 25-17, with 4 rows, has none.
- Of the 27,260 rows without a payload, most are era 0 (2,400 per metric), after the cache (512 per metric) or fCX with a pointer that isn't local (5,821, 1,058 and 6,767 in eras 1 to 3). One 2026-08-10 run has no `output_info` start. The no-pointer rows are all DS3 `no_result` with no value - elements the run didn't measure. No row is lost to the guard.


## 4. Seed error

The fit error only knows how well one curve fits. RB averages over random seeds, so the spread between seeds is another handle on the noise. For each target we drop one seed at a time, refit the seed-mean curve, and take the jackknife standard error of the $n$ fidelities $F_{(s)}$:

$$\sigma_\text{seed} = \sqrt{\frac{n-1}{n}\sum_{s=1}^{n}\left(F_{(s)} - \bar F\right)^2}$$

For RB1q the per-seed curve is `raw_output` itself. For RB2q we build each seed's $P_{00}$ from the single shots,

$$P_{00} = \tfrac14\left(1 + \langle z_c\rangle + \langle z_t\rangle + \langle z_c z_t\rangle\right)$$

which section 2 showed gives the stored curve. `seed_error` is NaN if the refit or any leave-one-out refit hits the grid edge. We compare it with `payload_error` (and DS3 `error` in eras 2 and 3), split by DS3 `value_status` - B4 keeps the `ok` rows, so that's where the ratio matters.


In [5]:
print("n_seeds by metric:", rb.groupby("metric")["n_seeds"].value_counts(dropna=False).to_dict())
print("seed_error NaN by metric:", rb["seed_error"].isna().groupby(rb["metric"]).sum().to_dict(),
      "| with refit_x0 NaN:", (rb["seed_error"].isna() & rb["refit_x0"].isna()).groupby(rb["metric"]).sum().to_dict(),
      "| with curve_match False or NA:", (rb["seed_error"].isna() & (rb["curve_match"] != True)).groupby(rb["metric"]).sum().to_dict())
print("seed_error quantiles by metric (payload targets):")
display(rb.groupby("metric")["seed_error"].quantile([0, 0.05, 0.5, 0.95, 1]).unstack())

rsub = sub[sub["metric"].isin(["fRB", "fCX"]) & sub["seed_error"].notna()].copy()
rsub["ratio"] = rsub["seed_error"] / rsub["payload_error"]
print("seed_error and seed_error / payload_error by metric and DS3 value_status:")
display(rsub.groupby(["metric", "value_status"]).agg(
    rows=("seed_error", "size"),
    seed_error_50=("seed_error", "median"), seed_error_95=("seed_error", lambda s: s.quantile(0.95)),
    ratio_50=("ratio", "median")).round(4))
print("rows with seed_error > 0.05, by metric and value_status:",
      rsub[rsub["seed_error"] > 0.05].groupby(["metric", "value_status"]).size().to_dict())

rok = rsub[rsub["value_status"] == "ok"]
print("ok rows only:")
ratios = pd.concat({
    "seed/payload_error": rok["ratio"].groupby([rok["metric"], rok["era"]]).describe(percentiles=[0.05, 0.5, 0.95]),
    "seed/ds3_error": (rok["seed_error"] / rok["error"]).groupby([rok["metric"], rok["era"]]).describe(percentiles=[0.05, 0.5, 0.95]),
})
display(ratios[["count", "5%", "50%", "95%"]].round(3))
print("share with seed_error > payload_error, by metric: ok rows",
      (rok["ratio"] > 1).groupby(rok["metric"]).mean().round(3).to_dict(),
      "| all rows", (rsub["ratio"] > 1).groupby(rsub["metric"]).mean().round(3).to_dict())


n_seeds by metric: {('fCX', 10.0): 3881, ('fCX', nan): 3, ('fRB', 10.0): 18274}


seed_error NaN by metric: {'fCX': 641, 'fRB': 666} | with refit_x0 NaN: {'fCX': 446, 'fRB': 222} | with curve_match False or NA: {'fCX': 3, 'fRB': 0}
seed_error quantiles by metric (payload targets):


,0.00,0.05,0.50,0.95,1.00
metric,,,,,
fCX,1.290176e-09,0.000861,0.00466,0.132676,0.813859
fRB,1.248875e-11,0.000193,0.00051,0.003825,0.646704


seed_error and seed_error / payload_error by metric and DS3 value_status:


rows  seed_error_50  seed_error_95  ratio_50
metric value_status                                               
fCX    low_r2          511         0.0831         0.2962    1.2080
       no_r2             1         0.0969         0.0969    1.4131
       ok             2474         0.0036         0.0402    1.8998
fRB    low_r2         1418         0.0016         0.0314    0.7224
       no_r2             2         0.0006         0.0007    2.3106
       ok            15096         0.0005         0.0025    1.6336

rows with seed_error > 0.05, by metric and value_status: {('fCX', 'low_r2'): 381, ('fCX', 'no_r2'): 1, ('fCX', 'ok'): 75, ('fRB', 'low_r2'): 44, ('fRB', 'ok'): 6}
ok rows only:


count     5%    50%    95%
                   metric era                                       
seed/payload_error fCX    era1_filter    1053.0  0.857  1.923  3.749
                          era2_ungated    174.0  0.825  1.877  3.164
                          era3_postproc  1247.0  0.770  1.881  3.631
                   fRB    era1_filter    6200.0  0.712  1.637  3.349
                          era2_ungated    991.0  0.702  1.600  3.347
                          era3_postproc  7905.0  0.726  1.634  3.442
seed/ds3_error     fCX    era1_filter       0.0    NaN    NaN    NaN
                          era2_ungated    174.0  0.825  1.877  3.164
                          era3_postproc  1247.0  0.770  1.881  3.631
                   fRB    era1_filter       0.0    NaN    NaN    NaN
                          era2_ungated    991.0  0.702  1.600  3.347
                          era3_postproc  7905.0  0.726  1.634  3.442

share with seed_error > payload_error, by metric: ok rows {'fCX': 0.901, 'fRB': 0.837} | all rows {'fCX': 0.854, 'fRB': 0.793}


**Results.**

- Every RB target has 10 seeds, apart from 3 fCX with none. `seed_error` is NaN for 666 fRB and 641 fCX targets.
- On `ok` rows the seed error is a median 1.60 to 1.64 (fRB) and 1.88 to 1.92 (fCX) times the fit error, steady across eras, and bigger in 0.837 and 0.901 of rows. Against DS3 `error` it's the same. So the fit error undersells the fidelity's uncertainty by roughly 1.6 to 1.9×.
- The long tail is the rejected fits. The median `seed_error` on fCX `low_r2` rows is 0.083 against 0.0036 on `ok`, and 381 of the 457 fCX rows above 0.05 are `low_r2`. On fRB `low_r2` rows the ratio even drops below 1 (0.72).


## 5. Strata columns

Last, the three columns B4 splits by. `rb_lengths` is the payload's lengths list. `coupling_retired` marks fCX rows after a coupling has stopped: a row is flagged when its run starts (its `output_info` start, else `created_at`) after the coupling's last guarded RB2q pointer, that pointer is earlier than the last one overall, and DS3 serves the coupling no fCX value after it. We check it against the 3a rule, the coupling's last `cr_u` row in the spine. `in_incident` marks rows whose `created_at` is inside a `periods.INCIDENTS` window.


In [6]:
lens = sub[sub["rb_lengths"].notna()]
display(lens.groupby(["metric", "rb_lengths"])["created_at"].agg(["size", "min", "max"]))
print("fRB/fCX rows with a payload value but no rb_lengths:",
      int((sub["metric"].isin(["fRB", "fCX"]) & sub["payload_value"].notna() & sub["rb_lengths"].isna()).sum()))

rb2q = links[(links["metric"] == "fCX") & links["guarded"]]
end = rb2q["measured_at"].max()
spine = pd.read_parquet(DEFAULT_SPINE, columns=["measurement_type", "measurement_name", "time_start"])
cr = spine[(spine["measurement_type"] == "CRUnitaryIZ") & spine["measurement_name"].str.match(r"cr_u_C\d+-T\d+$")]
cr_el = cr["measurement_name"].str.extract(r"C(\d+)-T(\d+)$").agg("-".join, axis=1)
print(f"last guarded RB2q pointer overall: {end}; last cr_u row in the spine: {cr['time_start'].max()}")
per = pd.DataFrame({
    "last_pointer": rb2q.groupby("element")["measured_at"].max(),
    "last_cr_u": cr.groupby(cr_el)["time_start"].max(),
    "last_ds3_value": fcx[fcx["value"].notna()].groupby("element")["created_at"].max(),
    "last_fcx_row": fcx.groupby("element")["created_at"].max(),
    "retired_rows": fcx.groupby("element")["coupling_retired"].sum(),
})
print("couplings whose last guarded RB2q pointer precedes the last overall:")
display(per[per["last_pointer"] < end].sort_values("last_pointer"))
print("couplings at the last pointer overall with no DS3 value after it:",
      per.index[(per["last_pointer"] == end) & ~(per["last_ds3_value"] > end)].tolist())

ret = fcx[fcx["coupling_retired"]]
start = sub.loc[fcx.index, "run_at"].fillna(fcx["created_at"])
print(f"coupling_retired rows: {len(ret):,} on {ret['element'].nunique()} couplings; with a DS3 value: {int(ret['value'].notna().sum())}")
print("retired rows whose run starts at or before the coupling's last cr_u row:",
      int((fcx["coupling_retired"] & (start <= fcx["element"].map(per["last_cr_u"]))).sum()))
tail = fcx[start > end]
print(f"fCX rows after the last pointer overall: {len(tail):,} ({tail['created_at'].min()} to {tail['created_at'].max()}), "
      f"with a DS3 value {int(tail['value'].notna().sum()):,}")

for inc in periods.INCIDENTS:
    inside = periods.within(sub["created_at"], inc.period)
    print(f"{inc.what}: {inc.period[0]} to {inc.period[1]}, rows {int(inside.sum()):,}, runs {sub.loc[inside, 'created_at'].nunique()}")
print("in_incident rows:", int(sub["in_incident"].sum()))

size                        min                        max
metric rb_lengths                                                                                                                                   
fCX    [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14]                                    1509 2024-10-15 19:39:16.280236 2026-08-16 21:53:56.049670
       [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 30, 40, 60, 80, 100]                       2065 2024-10-15 19:39:16.280236 2026-08-13 22:01:47.149851
fRB    [1, 27, 53, 79, 106, 132, 158, 184, 211, 237, 263, 289, 316, 342, 368, 394, 4...  17079 2024-10-15 19:39:16.280236 2026-08-16 21:53:56.049670

fRB/fCX rows with a payload value but no rb_lengths: 0


last guarded RB2q pointer overall: 2026-08-17 20:06:09; last cr_u row in the spine: 2026-08-18 09:12:51.648211
couplings whose last guarded RB2q pointer precedes the last overall:


,last_pointer,last_cr_u,last_ds3_value,last_fcx_row,retired_rows
1-10,2026-03-11 19:59:27,2026-03-24 16:11:26.810017,2026-03-11 21:25:15.156277,2026-03-11 21:25:15.156277,0
1-2,2026-03-11 19:59:27,2026-03-24 16:19:41.491115,2026-03-11 21:25:15.156277,2026-03-11 21:25:15.156277,0
2-9,2026-03-27 20:15:19,2026-03-28 20:07:43.376648,2026-03-26 16:12:17.975202,2026-09-14 22:01:36.723743,134
26-35,2026-03-28 20:13:14,2026-03-29 09:12:02.840715,2026-03-28 22:49:56.531215,2026-09-14 22:01:36.723743,133
19-11,2026-04-09 20:17:46,2026-04-10 20:11:30.820827,2026-04-09 23:01:48.905151,2026-09-14 22:01:36.723743,123
22-18,2026-04-28 20:15:44,2026-04-29 20:16:40.438775,2026-04-28 23:04:25.444745,2026-09-14 22:01:36.723743,105
10-11,2026-05-06 20:14:47,2026-05-07 09:08:03.811790,2026-05-06 22:53:20.645489,2026-09-14 22:01:36.723743,97
18-14,2026-07-13 20:05:50,2026-07-15 09:08:57.603090,2026-07-12 22:00:51.506905,2026-09-14 22:01:36.723743,44


couplings at the last pointer overall with no DS3 value after it: ['17-16', '7-16']
coupling_retired rows: 636 on 6 couplings; with a DS3 value: 0
retired rows whose run starts at or before the coupling's last cr_u row: 0
fCX rows after the last pointer overall: 512 (2026-08-19 22:08:30.334313 to 2026-09-14 22:01:36.723743), with a DS3 value 304
thermal cycle: 2026-03-09 08:45:16 to 2026-03-24 14:45:52, rows 128, runs 1
chip-wide f01 shift, fridge state unknown (DS4 gap): 2026-07-02 08:45:41 to 2026-07-05 21:54:09, rows 0, runs 0
in_incident rows: 128


**Results.**

- fRB has one lengths list (20 lengths up to 500). fCX has two - 14 lengths up to 14 on 1,509 rows and 17 up to 100 on 2,065 - and both run through the whole period. Every RB row with a payload has its list.
- Eight couplings stop before the last pointer, and six get flagged: 2-9, 26-35, 19-11, 22-18, 10-11 and 18-14, with last pointers from 2026-03-27 to 07-13. That's 636 rows, none with a DS3 value. Each one's last `cr_u` row is 0.5 to 1.6 days after its last pointer, and no flagged row starts before it, so this agrees with 3a. The other two, 1-10 and 1-2, have no fCX rows after 2026-03-11 at all.
- The last pointer overall (2026-08-17 20:06:09) is just the end of the `output_info` cache. The 512 fCX rows after it can't be judged - 17-16 and 7-16 have no DS3 value there and might have stopped too, but we don't flag them.
- `in_incident` covers 128 rows, all in the one run inside the thermal cycle. No run falls in the 2026-07-02 to 07-05 incident.


## Conclusion

| Result | Status |
|---|---|
| `kpi_supplemented` keeps every DS3 column of `kpi_enriched` and fills 32,512 `result_id` | DS3-only results carry over |
| Payload values and errors equal DS3 on every independent row, and we read the stored fits right | Payload columns are DS3's own fits |
| T1 and T2 are covered from era 1 on, fRB mostly, fCX 0.15 to 0.22 (the weekly RB2q sample), era 0 not at all | Fetch the rest of RB2q only if B4 needs fCX per run |
| The payload error is the only error in era 1; 6,019 era-1 fCX rows and all of era 0 have none | Use `error`, else `payload_error`, as `error_source` records |
| The `output_info` cache ends 2026-08-17, so later runs have no payload and can't be judged for retirement | Run payload-based results to the cache end |
| On `ok` rows the seed error is about 1.6× (fRB) and 1.9× (fCX) the fit error; its tail is `low_r2` fits | Report `seed_error` beside the fit error, on `ok` rows |
| Refits match the stored $x_0$, except where $x_0$ isn't pinned down (flat curves, stored $x_0 < 0$) | Count stored $x_0 < 0$ fits apart |
| fCX uses two lengths lists; six couplings are retired, matching the 3a `cr_u` rule; `in_incident` marks one thermal-cycle run | Strata columns ready for B4 |
